# 10. 오픈소스에서 배우는 Jev 패턴 (4) 브라우저 액션 선택

참고 프로젝트: [jev-browser](https://github.com/Ying-Kai-Liao/jev-browser)

## 배경

브라우저를 조작하는 에이전트는 보통 한 단계마다 큰 LLM 에 화면 전체를 보여 주고 "다음에 뭘 누를까"를 묻습니다. 화면 하나가 수천 토큰이고 작업 하나에 수십 단계가 걸리니 느리고 비쌉니다.

그런데 "다음에 뭘 누를까"는 화면에 있는 요소 중 하나를 고르는 일입니다. option 이 정해져 있습니다.

jev-browser 는 역할을 이렇게 나눕니다.

| | 하는 일 |
|---|---|
| LLM | 이루고 싶은 결과를 한 단계씩 말한다. 입력할 글자를 넘겨준다. |
| 코드 | 화면을 요소 목록으로 정리한다 (역할, 라벨, 상태, 보이는 글). |
| Jev | 어느 요소를 조작할지, 단계가 끝났는지, 오류가 났는지, 되돌릴 수 없는 조작인지 답한다. |

README 에 따르면 Jev 호출 한 번이 약 300ms 이고 한 단계에 2~4번 호출합니다.

## 직관

택시와 같습니다. 승객(LLM)은 "서울역 가 주세요"라고 목적지만 말합니다. 어느 차선으로 갈지, 지금 좌회전할지는 기사(Jev)가 매 순간 정합니다. 승객이 교차로마다 지시하지 않습니다.

```
LLM: "로그인한다" ──▶ 화면 요소 목록 ──▶ Jev ─┬─ target        어느 요소?
                                              ├─ done          이미 끝났나?
                                              ├─ has_error     오류가 보이나?
                                              └─ irreversible  되돌릴 수 없나?
```

네 판단을 합쳐 상태 하나를 만듭니다.

| 상태 | 조건 | 다음 행동 |
|---|---|---|
| `done` | 이미 끝남 (또는 누를 요소가 없고 끝났을 probability 가 절반 이상) | 다음 단계로 |
| `stuck` | 오류가 보이거나, 누를 요소가 없거나, 확신이 낮음 | LLM 에 다시 계획 요청 |
| `needs_confirmation` | 되돌릴 수 없는 조작 | 사람 확인 |
| `act` | 그 외 | 바로 실행 |

이 노트북은 실제 브라우저 없이 화면을 요소 목록으로 흉내 냅니다.

In [1]:
import sys
sys.path.insert(0, "..")  # 프로젝트 루트의 jev_agent 패키지를 불러오기 위함

import pandas as pd
from dotenv import load_dotenv
load_dotenv("../.env")

from jev_agent.jev import JevClient, choice, noul, score
from jev_agent.patterns import run_pattern

pd.set_option("display.max_colwidth", 60)
jev = JevClient()

In [2]:
def rows_table(output: dict) -> pd.DataFrame:
    """패턴 결과의 rows 를 표로 바꾼다. probability 는 큰 순서로 세 개까지 한 칸에 적는다."""
    records = []
    for item in output["rows"]:
        top = sorted(item["probabilities"].items(), key=lambda pair: -pair[1])[:3]
        records.append({
            "항목": item["title"],
            "판정": item["verdict"],
            "probability": ", ".join(f"{name} {value:.2f}" for name, value in top),
        })
    return pd.DataFrame(records)

## Jev 에 무엇을 보내는가

화면 요소가 그대로 `choice` 의 option 이 됩니다. option 에 없는 요소는 고를 수 없으므로, 없는 버튼을 누르려는 일이 생기지 않습니다. 누를 것이 없을 때를 위한 `none` option 이 하나 더 있습니다.

In [3]:
from jev_agent.patterns import actions

page = actions.SAMPLE_PAGES[0]
state, questions = actions.plan(page)

print("step_goal:", state["step_goal"])
print("elements :", state["elements"])
print("target option:", list(questions["target"]["criteria"]))

step_goal: 이메일과 비밀번호를 입력했으니 로그인한다
elements : ["e1: textbox '이메일'", "e2: textbox '비밀번호'", "e3: button '로그인'", "e4: link '비밀번호 찾기'", "e5: button '회원가입'"]
target option: ['e1', 'e2', 'e3', 'e4', 'e5', 'none']


## 화면 네 개에서 실행

In [4]:
records = []
for page in actions.SAMPLE_PAGES:
    output = run_pattern(jev, "browser_action", page)
    by_title = {item["title"]: list(item["probabilities"].values())[0] for item in output["rows"][1:]}
    records.append({
        "단계 목표": page["step_goal"],
        "화면": page["page_title"],
        "고른 요소": output["rows"][0]["verdict"],
        "완료": round(by_title["이미 완료됨"], 2),
        "오류": round(by_title["오류 표시"], 2),
        "비가역": round(by_title["되돌릴 수 없음"], 2),
        "상태": output["status"],
        "지연(ms)": output["latency_ms"],
    })
pd.DataFrame(records)

,단계 목표,화면,고른 요소,완료,오류,비가역,상태,지연(ms)
0,이메일과 비밀번호를 입력했으니 로그인한다,로그인,e3 · 로그인,0.20,0.04,0.07,act,252
1,장바구니의 상품을 주문한다,주문/결제,"e2 · 89,000원 결제하기",0.07,0.04,0.89,needs_confirmation,241
2,로그인한다,마이페이지,none · 없음,0.93,0.03,0.07,done,333
3,로그인한다,로그인,e1 · 비밀번호,0.04,0.99,0.06,stuck,339


네 화면이 네 가지 상태를 하나씩 보여 주는지 확인하세요. 결제 버튼은 고를 요소가 분명하더라도 바로 누르지 않고 `needs_confirmation` 으로 멈춰야 합니다.

## 작은 루프로 묶기

이제 LLM 과 Jev 를 묶어 봅니다. LLM 은 처음에 한 번만 호출해서 단계를 나누고, 그 뒤 화면 조작은 전부 Jev 가 정합니다.

화면은 아래처럼 흉내 냅니다. `next` 는 그 요소를 누르면 어느 화면으로 가는지입니다.

In [5]:
SITE = {
    "login": {
        "page_title": "로그인",
        "visible_text": "이메일: teddy@example.com  비밀번호: ********",
        "elements": [
            {"id": "e1", "role": "button", "label": "로그인"},
            {"id": "e2", "role": "link", "label": "비밀번호 찾기"},
        ],
        "next": {"e1": "home"},
    },
    "home": {
        "page_title": "마이페이지",
        "visible_text": "테디님, 환영합니다.  적립금 3,200P",
        "elements": [
            {"id": "e1", "role": "link", "label": "주문 내역"},
            {"id": "e2", "role": "link", "label": "장바구니 (1)"},
            {"id": "e3", "role": "button", "label": "로그아웃"},
        ],
        "next": {"e2": "cart"},
    },
    "cart": {
        "page_title": "장바구니",
        "visible_text": "러닝화 270mm 1개  합계 89,000원  결제 수단: 신용카드",
        "elements": [
            {"id": "e1", "role": "button", "label": "쿠폰 적용"},
            {"id": "e2", "role": "button", "label": "89,000원 결제하기"},
        ],
        "next": {},
    },
}

In [6]:
from pydantic import BaseModel, Field
from jev_agent.agent import build_chat_model


class BrowserPlan(BaseModel):
    steps: list[str] = Field(description="각 단계에서 이루려는 결과를 한 문장씩. 어느 버튼을 누를지는 적지 않는다.")


planner = build_chat_model().with_structured_output(BrowserPlan)
plan = planner.invoke(
    "쇼핑몰 웹사이트에서 '로그인한 뒤 장바구니에 담긴 상품을 결제'하려고 합니다. "
    "지금은 로그인 화면이고 이메일과 비밀번호는 이미 입력되어 있습니다. "
    "이 작업을 정확히 3단계로 나누어 주세요. 각 단계는 '~한다'로 끝나는 15자 안팎의 짧은 문장 하나로, "
    "이루려는 일 하나만 적습니다. 어떤 버튼을 누르는지나 확인 절차는 적지 않습니다. "
    "형식 예: '로그인한다', '장바구니 화면으로 이동한다', '장바구니의 상품을 결제한다'."
)
plan.steps

['로그인한다', '장바구니로 이동한다', '장바구니 상품을 결제한다']

In [7]:
MAX_ACTIONS_PER_STEP = 3  # 한 단계에서 조작을 반복할 수 있는 횟수의 상한

current = "login"
jev_calls, jev_ms = 0, 0
for step in plan.steps:
    print(f"\n단계: {step}")
    for _ in range(MAX_ACTIONS_PER_STEP):
        screen = SITE[current]
        output = run_pattern(jev, "browser_action", {"step_goal": step, **screen})
        jev_calls, jev_ms = jev_calls + 1, jev_ms + output["latency_ms"]
        print(f"  [{screen['page_title']}] 상태={output['status']}  대상={output['rows'][0]['verdict']}")
        if output["status"] != "act":
            break
        current = screen["next"].get(output["target"], current)
    if output["status"] in ("stuck", "needs_confirmation"):
        print(f"  → 여기서 멈춥니다: {output['stats'][1]['value']}")
        break

print(f"\nLLM 호출 1회(계획), Jev 호출 {jev_calls}회, Jev 지연 합계 {jev_ms}ms")


단계: 로그인한다


  [로그인] 상태=act  대상=e1 · 로그인


  [마이페이지] 상태=done  대상=none · 없음

단계: 장바구니로 이동한다


  [마이페이지] 상태=act  대상=e2 · 장바구니 (1)


  [장바구니] 상태=done  대상=none · 없음

단계: 장바구니 상품을 결제한다


  [장바구니] 상태=needs_confirmation  대상=e2 · 89,000원 결제하기
  → 여기서 멈춥니다: 되돌릴 수 없는 조작, 사람 확인 필요

LLM 호출 1회(계획), Jev 호출 5회, Jev 지연 합계 1327ms


LLM 이 적은 단계 문장은 실행마다 조금씩 달라지므로 위 출력도 달라질 수 있습니다.

단계 문장의 모양이 Jev 판단에 영향을 줍니다. 이 노트북을 만들면서 확인한 바로는 "로그인한다"처럼 짧게 할 일을 적으면 요소 선택 confidence 가 0.99 였고, "로그인이 완료되어 계정에 접속한 상태다"처럼 도달 상태를 길게 적으면 0.49 로 떨어져 `stuck` 이 됐습니다. 그래서 위 프롬프트는 LLM 에 문장 형식을 정해 줍니다. Jev 에 넘기는 글은 사람이 읽기 좋은 문장보다 판단하기 쉬운 문장이어야 합니다.

확인할 것은 두 가지입니다. 화면 이동은 Jev 판단만으로 이뤄졌는지, 그리고 결제 버튼 앞에서 멈췄는지입니다.

## 정리

- LLM 은 목표를 말하고 Jev 는 조작을 고릅니다. LLM 호출 횟수가 단계 수와 무관해집니다.
- 화면 요소가 그대로 option 이 되므로 없는 요소를 누르려는 일이 생기지 않습니다.
- 완료 판정을 따로 묻는 것이 중요합니다. 에이전트가 끝나지 않았는데 끝났다고 하는 것이 가장 위험한 실패이기 때문입니다. jev-browser README 는 실제 작업 42건 중 40건 성공, 잘못된 완료 선언 0건이라고 적고 있습니다 (README 에 적힌 수치이며 이 튜토리얼에서 재현하지 않았습니다).
- 한계도 분명합니다. 화면을 요소 목록으로 잘 정리하는 코드가 먼저 있어야 하고, 글자를 입력하는 일은 여전히 LLM 이 값을 줘야 합니다.

---

## 시리즈를 마치며: 나머지 세 프로젝트

블로그 글의 8개 중 세 개는 노트북으로 만들지 않았습니다.

**[NanoJev](https://github.com/TianyuCodings/NanoJev), [open-alternative-jev](https://github.com/ikermoel/open-alternative-jev).** Jev 같은 판단 모델을 직접 만들거나 공개 가중치 모델(Qwen 등)로 재현하는 프로젝트입니다. 핵심 생각은 같습니다. 텍스트를 생성하지 않고, option 각각에 대한 probability 를 바로 읽어 냅니다. 로컬 GPU 와 모델 가중치가 필요해 이 튜토리얼에서는 실행하지 않았습니다. API 에 데이터를 보낼 수 없는 환경이라면 이쪽이 대안입니다.

**[Reticle](https://github.com/reticlehq/reticle).** 에이전트가 만든 코드를 실제로 실행하면서 화면, 네트워크, 콘솔을 관찰해 검증하는 도구입니다. 블로그 글에 따르면 Jev 연동은 아직 계획 단계입니다. 판단 모델과는 별개로, "에이전트의 말이 아니라 실제 동작으로 확인한다"는 방향이 이 시리즈의 주제와 닿아 있습니다.

## 네 가지 패턴에 공통된 것

| | Jev 에 보내는 것 | 받는 판단 | LLM 이 여전히 하는 일 |
|---|---|---|---|
| 07 메모리 압축 | 결과를 뺀 대화 요약 | 호출별 keep probability 2개 | 줄어든 대화로 작업 계속 |
| 08 도구 게이트 | 명령 또는 출력 | 위험 4종, 실패 유형 | 명령 작성, 조언 읽고 대응 |
| 09 라우팅 | 질의와 후보 결과 | 소스, 기간, 난이도, 관련도 | 고른 결과로 답변 작성 |
| 10 브라우저 액션 | 단계 목표와 요소 목록 | 대상, 완료, 오류, 비가역 | 단계 계획, 입력값 작성 |

모두 같은 구조입니다. **option 을 코드로 만들고, Jev 가 고르고, probability 와 기준값으로 분기합니다.** 문장을 쓰는 일만 LLM 에 남깁니다.